In [68]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from pathlib import Path

In [69]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def to_tensor(x: np.ndarray | torch.Tensor, device: torch.device, dtype=torch.float32) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.tensor(x, dtype=dtype, device=device)

In [70]:
def save_checkpoint(path: str, payload: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)

def load_checkpoint(path: str, device: str = "cpu") -> dict:
    return torch.load(path, map_location=device)

In [71]:
@dataclass
class MarketConfig:
    dt: float = 0.2
    gamma: float = 0.99
    lam: float = 0.05
    i_min: float = -10.0
    i_max: float = 10.0
    window: int = 10  # W
    device: str = "gpu"

    # Choix du scénario:
    # "theta"                 -> theta_t markovien, kappa/sigma constants
    # "theta_kappa"           -> theta_t et kappa_t markoviens
    # "theta_kappa_sigma"     -> theta_t, kappa_t, sigma_t markoviens
    scenario: str = "theta"

    theta_values: tuple = (0.9, 1.0, 1.1)
    kappa_values: tuple = (3.0, 7.0)
    sigma_values: tuple = (0.1, 0.3)

    theta_transition: tuple = (
        (0.90, 0.05, 0.05),
        (0.05, 0.90, 0.05),
        (0.05, 0.05, 0.90),
    )
    kappa_transition: tuple = (
        (0.90, 0.10),
        (0.10, 0.90),
    )
    sigma_transition: tuple = (
        (0.90, 0.10),
        (0.10, 0.90),
    )

In [72]:
@dataclass
class TrainingConfig:
    batch_size: int = 256
    train_steps_supervised: int = 2000
    train_steps_rl: int = 3000
    test_episodes: int = 200
    episode_length: int = 500

    lr_supervised: float = 1e-3
    lr_actor: float = 1e-3
    lr_critic: float = 1e-3
    lr_aux: float = 1e-3

    hidden_size_gru: int = 20
    gru_layers: int = 1

    actor_hidden: int = 128
    critic_hidden: int = 128

    tau_target: float = 1e-3
    exploration_a: float = 100.0
    exploration_min: float = 0.05

    critic_updates_per_step: int = 1
    actor_updates_per_step: int = 1

    # normalisation simple des features
    state_norm_eps: float = 1e-8

In [73]:
class MarkovChain:
    def __init__(self, transition: np.ndarray):
        self.transition = transition
        self.n_states = transition.shape[0]

    def sample_initial(self, batch_size: int) -> np.ndarray:
        probs = np.ones(self.n_states) / self.n_states
        return np.random.choice(self.n_states, size=batch_size, p=probs)

    def step(self, current_state: np.ndarray) -> np.ndarray:
        next_state = np.empty_like(current_state)
        for i, s in enumerate(current_state):
            next_state[i] = np.random.choice(self.n_states, p=self.transition[s])
        return next_state

In [74]:
class OUSimulator:
    def __init__(self, market_cfg: MarketConfig):
        self.cfg = market_cfg
        self.device = torch.device(market_cfg.device)

        self.theta_mc = MarkovChain(np.array(market_cfg.theta_transition, dtype=float))
        self.kappa_mc = MarkovChain(np.array(market_cfg.kappa_transition, dtype=float))
        self.sigma_mc = MarkovChain(np.array(market_cfg.sigma_transition, dtype=float))

    def _scenario_params(
        self,
        theta_idx: np.ndarray,
        kappa_idx: np.ndarray,
        sigma_idx: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        theta_vals = np.array(self.cfg.theta_values, dtype=float)[theta_idx]

        if self.cfg.scenario == "theta":
            kappa_vals = np.full_like(theta_vals, 5.0, dtype=float)
            sigma_vals = np.full_like(theta_vals, 0.2, dtype=float)

        elif self.cfg.scenario == "theta_kappa":
            kappa_vals = np.array(self.cfg.kappa_values, dtype=float)[kappa_idx]
            sigma_vals = np.full_like(theta_vals, 0.2, dtype=float)

        elif self.cfg.scenario == "theta_kappa_sigma":
            kappa_vals = np.array(self.cfg.kappa_values, dtype=float)[kappa_idx]
            sigma_vals = np.array(self.cfg.sigma_values, dtype=float)[sigma_idx]

        else:
            raise ValueError(f"Unknown scenario: {self.cfg.scenario}")

        return theta_vals, kappa_vals, sigma_vals

    def _sample_initial_signal(
        self,
        theta0: np.ndarray,
        kappa0: np.ndarray,
        sigma0: np.ndarray,
    ) -> np.ndarray:
        """
        Approximation simple de l'initialisation stationnaire.
        OU stationnaire: Var = sigma^2/(2kappa)
        """
        mu_inv = theta0
        var_inv = (sigma0 ** 2) / (2.0 * np.maximum(kappa0, 1e-6))
        std_init = np.sqrt(3.0 * var_inv)
        return np.random.normal(loc=mu_inv, scale=std_init)

    def sample_batch(self, batch_size: int) -> Dict[str, torch.Tensor]:
        W = self.cfg.window
        T = W + 2  # S_{t-W}, ..., S_t, S_{t+1}
        dt = self.cfg.dt

        theta_idx = np.zeros((batch_size, T), dtype=int)
        kappa_idx = np.zeros((batch_size, T), dtype=int)
        sigma_idx = np.zeros((batch_size, T), dtype=int)

        theta_idx[:, 0] = self.theta_mc.sample_initial(batch_size)
        kappa_idx[:, 0] = self.kappa_mc.sample_initial(batch_size)
        sigma_idx[:, 0] = self.sigma_mc.sample_initial(batch_size)

        for t in range(T - 1):
            theta_idx[:, t + 1] = self.theta_mc.step(theta_idx[:, t])

            if self.cfg.scenario in ("theta_kappa", "theta_kappa_sigma"):
                kappa_idx[:, t + 1] = self.kappa_mc.step(kappa_idx[:, t])
            else:
                kappa_idx[:, t + 1] = kappa_idx[:, t]

            if self.cfg.scenario == "theta_kappa_sigma":
                sigma_idx[:, t + 1] = self.sigma_mc.step(sigma_idx[:, t])
            else:
                sigma_idx[:, t + 1] = sigma_idx[:, t]

        theta0, kappa0, sigma0 = self._scenario_params(
            theta_idx[:, 0], kappa_idx[:, 0], sigma_idx[:, 0]
        )

        S = np.zeros((batch_size, T), dtype=np.float32)
        S[:, 0] = self._sample_initial_signal(theta0, kappa0, sigma0).astype(np.float32)

        for t in range(T - 1):
            theta_t, kappa_t, sigma_t = self._scenario_params(
                theta_idx[:, t], kappa_idx[:, t], sigma_idx[:, t]
            )
            eps = np.random.randn(batch_size).astype(np.float32)
            S[:, t + 1] = (
                S[:, t]
                + kappa_t * (theta_t - S[:, t]) * dt
                + sigma_t * np.sqrt(dt) * eps
            ).astype(np.float32)

        I_t = np.random.uniform(self.cfg.i_min, self.cfg.i_max, size=(batch_size, 1)).astype(np.float32)

        X_t = S[:, :W + 1]       # S_{t-W}, ..., S_t
        X_tp1 = S[:, 1:W + 2]    # S_{t-W+1}, ..., S_{t+1}
        S_t = S[:, W:W + 1]
        S_tp1 = S[:, W + 1:W + 2]

        out = {
            "X_t": to_tensor(X_t, self.device),
            "X_tp1": to_tensor(X_tp1, self.device),
            "S_t": to_tensor(S_t, self.device),
            "S_tp1": to_tensor(S_tp1, self.device),
            "I_t": to_tensor(I_t, self.device),
            "theta_label_t": to_tensor(theta_idx[:, W], self.device, dtype=torch.long),
            "theta_label_tp1": to_tensor(theta_idx[:, W + 1], self.device, dtype=torch.long),
        }
        return out

    def sample_episode(self, length: int) -> Dict[str, np.ndarray]:
        """
        Génère un long épisode pour évaluation.
        """
        W = self.cfg.window
        total = length + W + 1

        theta_idx = np.zeros(total, dtype=int)
        kappa_idx = np.zeros(total, dtype=int)
        sigma_idx = np.zeros(total, dtype=int)

        theta_idx[0] = self.theta_mc.sample_initial(1)[0]
        kappa_idx[0] = self.kappa_mc.sample_initial(1)[0]
        sigma_idx[0] = self.sigma_mc.sample_initial(1)[0]

        for t in range(total - 1):
            theta_idx[t + 1] = self.theta_mc.step(np.array([theta_idx[t]]))[0]

            if self.cfg.scenario in ("theta_kappa", "theta_kappa_sigma"):
                kappa_idx[t + 1] = self.kappa_mc.step(np.array([kappa_idx[t]]))[0]
            else:
                kappa_idx[t + 1] = kappa_idx[t]

            if self.cfg.scenario == "theta_kappa_sigma":
                sigma_idx[t + 1] = self.sigma_mc.step(np.array([sigma_idx[t]]))[0]
            else:
                sigma_idx[t + 1] = sigma_idx[t]

        theta0, kappa0, sigma0 = self._scenario_params(
            np.array([theta_idx[0]]),
            np.array([kappa_idx[0]]),
            np.array([sigma_idx[0]]),
        )
        S = np.zeros(total, dtype=np.float32)
        S[0] = self._sample_initial_signal(theta0, kappa0, sigma0)[0].astype(np.float32)

        for t in range(total - 1):
            theta_t, kappa_t, sigma_t = self._scenario_params(
                np.array([theta_idx[t]]),
                np.array([kappa_idx[t]]),
                np.array([sigma_idx[t]]),
            )
            eps = np.random.randn()
            S[t + 1] = (
                S[t]
                + kappa_t[0] * (theta_t[0] - S[t]) * self.cfg.dt
                + sigma_t[0] * np.sqrt(self.cfg.dt) * eps
            ).astype(np.float32)

        return {
            "S": S,
            "theta_idx": theta_idx,
            "kappa_idx": kappa_idx,
            "sigma_idx": sigma_idx,
        }

In [75]:
class GRUEncoder(nn.Module):
    def __init__(self, hidden_size: int, num_layers: int = 1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [B, W+1]
        x = x.unsqueeze(-1)  # [B, W+1, 1]
        out, h_n = self.gru(x)
        return h_n[-1]  # [B, hidden_size]


class PriceHead(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.SiLU(),
            nn.Linear(in_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [76]:
class PriceRegressor(nn.Module):
    def __init__(self, hidden_size: int, num_layers: int):
        super().__init__()
        self.encoder = GRUEncoder(hidden_size, num_layers)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        return self.head(h)

In [77]:
class RegimeClassifier(nn.Module):
    def __init__(self, hidden_size: int, num_layers: int, n_classes: int):
        super().__init__()
        self.encoder = GRUEncoder(hidden_size, num_layers)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        logits = self.head(h)
        return logits

    def probs(self, x: torch.Tensor) -> torch.Tensor:
        return F.softmax(self.forward(x), dim=-1)

class Actor(nn.Module):
    def __init__(self, state_dim: int, hidden_dim: int, i_max: float):
        super().__init__()
        self.i_max = i_max
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
            nn.Tanh(),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.i_max * self.net(state)


class Critic(nn.Module):
    def __init__(self, state_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        x = torch.cat([state, action], dim=-1)
        return self.net(x)

In [78]:
class RunningNormalizer:
    def __init__(self, dim: int, eps: float = 1e-8, momentum: float = 0.01):
        self.dim = dim
        self.eps = eps
        self.momentum = momentum
        self.mean = None
        self.var = None

    def update(self, x: torch.Tensor) -> None:
        with torch.no_grad():
            batch_mean = x.mean(dim=0, keepdim=True)
            batch_var = x.var(dim=0, unbiased=False, keepdim=True)

            if self.mean is None:
                self.mean = batch_mean
                self.var = batch_var
            else:
                self.mean = (1 - self.momentum) * self.mean + self.momentum * batch_mean
                self.var = (1 - self.momentum) * self.var + self.momentum * batch_var

    def normalize(self, x: torch.Tensor) -> torch.Tensor:
        if self.mean is None or self.var is None:
            return x
        return (x - self.mean) / torch.sqrt(self.var + self.eps)

In [79]:
def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    for p_tgt, p_src in zip(target.parameters(), source.parameters()):
        p_tgt.data.copy_(tau * p_src.data + (1.0 - tau) * p_tgt.data)

def exploration_noise(step: int, a: float, eps_min: float, device: torch.device) -> torch.Tensor:
    eps = max(a / (a + step), eps_min)
    return torch.tensor(eps, dtype=torch.float32, device=device)

def trading_reward(
    i_tp1: torch.Tensor,
    i_t: torch.Tensor,
    s_t: torch.Tensor,
    s_tp1: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    return i_tp1 * (s_tp1 - s_t) - lam * torch.abs(i_tp1 - i_t)

In [80]:
class ProbPretrainer:
    def __init__(self, model: RegimeClassifier, simulator: OUSimulator, train_cfg: TrainingConfig):
        self.model = model
        self.simulator = simulator
        self.cfg = train_cfg
        self.device = next(model.parameters()).device
        self.opt = Adam(self.model.parameters(), lr=self.cfg.lr_supervised)

    def train(self) -> None:
        self.model.train()
        for step in range(1, self.cfg.train_steps_supervised + 1):
            batch = self.simulator.sample_batch(self.cfg.batch_size)
            x_t = batch["X_t"]
            y = batch["theta_label_t"]

            logits = self.model(x_t)
            loss = F.cross_entropy(logits, y)

            self.opt.zero_grad()
            loss.backward()
            self.opt.step()

            if step % 200 == 0:
                acc = (logits.argmax(dim=-1) == y).float().mean().item()
                print(f"[Prob pretrain] step={step:4d} loss={loss.item():.4f} acc={acc:.4f}")


class RegPretrainer:
    def __init__(self, model, simulator: OUSimulator, train_cfg: TrainingConfig): #PriceRegressor
        self.model = model
        self.simulator = simulator
        self.cfg = train_cfg
        self.device = next(model.parameters()).device
        self.opt = Adam(self.model.parameters(), lr=self.cfg.lr_supervised)

    def train(self) -> None:
        self.model.train()
        for step in range(1, self.cfg.train_steps_supervised + 1):
            batch = self.simulator.sample_batch(self.cfg.batch_size)
            x_t = batch["X_t"]
            s_tp1 = batch["S_tp1"]

            pred = self.model(x_t)
            loss = F.mse_loss(pred, s_tp1)

            self.opt.zero_grad()
            loss.backward()
            self.opt.step()

            if step % 200 == 0:
                print(f"[Reg pretrain]  step={step:4d} mse={loss.item():.6f}")

In [81]:
class BaseDDPGTrainer:
    def __init__(self, market_cfg: MarketConfig, train_cfg: TrainingConfig):
        self.mcfg = market_cfg
        self.tcfg = train_cfg
        self.device = torch.device(market_cfg.device)
        self.simulator = OUSimulator(market_cfg)

    def _build_state_normalizer(self, state_dim: int) -> RunningNormalizer:
        return RunningNormalizer(state_dim, eps=self.tcfg.state_norm_eps)

    def evaluate_policy(self, actor: Actor, state_builder_fn, name: str) -> None:
        actor.eval()
        rewards = []

        with torch.no_grad():
            for _ in range(self.tcfg.test_episodes):
                episode = self.simulator.sample_episode(self.tcfg.episode_length)
                S = episode["S"]

                I_t = 0.0
                cum_reward = 0.0
                W = self.mcfg.window

                for t in range(W, W + self.tcfg.episode_length):
                    X_t_np = S[t - W:t + 1][None, :]
                    S_t_np = np.array([[S[t]]], dtype=np.float32)
                    S_tp1_np = np.array([[S[t + 1]]], dtype=np.float32)
                    I_t_np = np.array([[I_t]], dtype=np.float32)

                    batch = {
                        "X_t": to_tensor(X_t_np, self.device),
                        "S_t": to_tensor(S_t_np, self.device),
                        "S_tp1": to_tensor(S_tp1_np, self.device),
                        "I_t": to_tensor(I_t_np, self.device),
                    }

                    state = state_builder_fn(batch)
                    i_tp1 = actor(state).clamp(self.mcfg.i_min, self.mcfg.i_max)

                    r = trading_reward(
                        i_tp1=i_tp1,
                        i_t=batch["I_t"],
                        s_t=batch["S_t"],
                        s_tp1=batch["S_tp1"],
                        lam=self.mcfg.lam,
                    )
                    cum_reward += float(r.item())
                    I_t = float(i_tp1.item())

                rewards.append(cum_reward)

        rewards = np.array(rewards)
        print(
            f"[{name} evaluation] mean={rewards.mean():.4f} "
            f"std={rewards.std():.4f} min={rewards.min():.4f} max={rewards.max():.4f}"
        )

In [82]:
class HidDDPGTrainer(BaseDDPGTrainer):
    def __init__(self, market_cfg: MarketConfig, train_cfg: TrainingConfig):
        super().__init__(market_cfg, train_cfg)

        h = train_cfg.hidden_size_gru
        self.encoder = GRUEncoder(h, train_cfg.gru_layers).to(self.device)
        self.price_head = PriceHead(h).to(self.device)

        state_dim = h + 2  # o_t + S_t + I_t
        self.normalizer = self._build_state_normalizer(state_dim)

        self.actor = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic = Critic(state_dim, train_cfg.critic_hidden).to(self.device)

        self.actor_tgt = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic_tgt = Critic(state_dim, train_cfg.critic_hidden).to(self.device)
        self.actor_tgt.load_state_dict(self.actor.state_dict())
        self.critic_tgt.load_state_dict(self.critic.state_dict())

        self.opt_aux = Adam(
            list(self.encoder.parameters()) + list(self.price_head.parameters()),
            lr=train_cfg.lr_aux,
        )
        self.opt_actor = Adam(self.actor.parameters(), lr=train_cfg.lr_actor)
        self.opt_critic = Adam(self.critic.parameters(), lr=train_cfg.lr_critic)

    def build_state(self, batch: Dict[str, torch.Tensor], next_window: bool = False) -> torch.Tensor:
        x = batch["X_tp1"] if next_window else batch["X_t"]
        s = batch["S_tp1"] if next_window else batch["S_t"]
        i = batch["I_tp1"] if next_window else batch["I_t"]

        o = self.encoder(x)
        state = torch.cat([o, s, i], dim=-1)
        self.normalizer.update(state.detach())
        return self.normalizer.normalize(state).detach()

    def train(self) -> None:
        for step in range(1, self.tcfg.train_steps_rl + 1):
            batch = self.simulator.sample_batch(self.tcfg.batch_size)

            # ------------------------------------------------
            # 1) Tâche auxiliaire GRU -> prédire S_{t+1}
            # ------------------------------------------------
            self.encoder.train()
            self.price_head.train()

            o_t = self.encoder(batch["X_t"])
            s_hat_tp1 = self.price_head(o_t)
            loss_aux = F.mse_loss(s_hat_tp1, batch["S_tp1"])

            self.opt_aux.zero_grad()
            loss_aux.backward()
            self.opt_aux.step()

            # ------------------------------------------------
            # 2) État courant
            # ------------------------------------------------
            state_t = self.build_state(batch, next_window=False)

            # action avec bruit
            self.actor.train()
            self.critic.train()

            eps = exploration_noise(step, self.tcfg.exploration_a, self.tcfg.exploration_min, self.device)
            with torch.no_grad():
                a_noise = eps * torch.randn((self.tcfg.batch_size, 1), device=self.device)
                i_tp1 = self.actor(state_t) + a_noise
                i_tp1 = i_tp1.clamp(self.mcfg.i_min, self.mcfg.i_max)

            reward = trading_reward(
                i_tp1=i_tp1,
                i_t=batch["I_t"],
                s_t=batch["S_t"],
                s_tp1=batch["S_tp1"],
                lam=self.mcfg.lam,
            )

            batch["I_tp1"] = i_tp1.detach()

            # ------------------------------------------------
            # 3) État suivant
            # ------------------------------------------------
            with torch.no_grad():
                state_tp1 = self.build_state(batch, next_window=True)
                a_tp1 = self.actor_tgt(state_tp1).clamp(self.mcfg.i_min, self.mcfg.i_max)
                y = reward + self.mcfg.gamma * self.critic_tgt(state_tp1, a_tp1)

            # ------------------------------------------------
            # 4) Critic
            # ------------------------------------------------
            for _ in range(self.tcfg.critic_updates_per_step):
                q = self.critic(state_t, i_tp1.detach())
                loss_critic = F.mse_loss(q, y)

                self.opt_critic.zero_grad()
                loss_critic.backward()
                self.opt_critic.step()

            # ------------------------------------------------
            # 5) Actor
            # ------------------------------------------------
            for _ in range(self.tcfg.actor_updates_per_step):
                a = self.actor(state_t).clamp(self.mcfg.i_min, self.mcfg.i_max)
                loss_actor = -self.critic(state_t.detach(), a).mean()

                self.opt_actor.zero_grad()
                loss_actor.backward()
                self.opt_actor.step()

            # ------------------------------------------------
            # 6) Target update
            # ------------------------------------------------
            soft_update(self.actor_tgt, self.actor, self.tcfg.tau_target)
            soft_update(self.critic_tgt, self.critic, self.tcfg.tau_target)

            if step % 200 == 0:
                print(
                    f"[hid-DDPG] step={step:4d} aux={loss_aux.item():.6f} "
                    f"critic={loss_critic.item():.6f} actor={loss_actor.item():.6f} "
                    f"reward_mean={reward.mean().item():.4f}"
                )

        self.evaluate_policy(self.actor, self._eval_state_builder, "hid-DDPG")

    def _eval_state_builder(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        self.encoder.eval()
        o = self.encoder(batch["X_t"])
        state = torch.cat([o, batch["S_t"], batch["I_t"]], dim=-1)
        return self.normalizer.normalize(state)

    def save(self, path: str) -> None:
        save_checkpoint(
            path,
            {
                "encoder": self.encoder.state_dict(),
                "price_head": self.price_head.state_dict(),
                "actor": self.actor.state_dict(),
                "critic": self.critic.state_dict(),
                "actor_tgt": self.actor_tgt.state_dict(),
                "critic_tgt": self.critic_tgt.state_dict(),
                "opt_aux": self.opt_aux.state_dict(),
                "opt_actor": self.opt_actor.state_dict(),
                "opt_critic": self.opt_critic.state_dict(),
                "market_cfg": self.mcfg.__dict__,
                "train_cfg": self.tcfg.__dict__,
            },
        )

    def load(self, path: str) -> None:
        ckpt = load_checkpoint(path, device=self.device)
        self.encoder.load_state_dict(ckpt["encoder"])
        self.price_head.load_state_dict(ckpt["price_head"])
        self.actor.load_state_dict(ckpt["actor"])
        self.critic.load_state_dict(ckpt["critic"])
        self.actor_tgt.load_state_dict(ckpt["actor_tgt"])
        self.critic_tgt.load_state_dict(ckpt["critic_tgt"])
        self.opt_aux.load_state_dict(ckpt["opt_aux"])
        self.opt_actor.load_state_dict(ckpt["opt_actor"])
        self.opt_critic.load_state_dict(ckpt["opt_critic"])

In [83]:
class ProbDDPGTrainer(BaseDDPGTrainer):
    def __init__(self, market_cfg: MarketConfig, train_cfg: TrainingConfig):
        super().__init__(market_cfg, train_cfg)

        self.filter_model = RegimeClassifier(
            hidden_size=train_cfg.hidden_size_gru,
            num_layers=train_cfg.gru_layers,
            n_classes=len(market_cfg.theta_values),
        ).to(self.device)

        state_dim = 2 + len(market_cfg.theta_values)  # S_t + I_t + probs
        self.normalizer = self._build_state_normalizer(state_dim)

        self.actor = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic = Critic(state_dim, train_cfg.critic_hidden).to(self.device)

        self.actor_tgt = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic_tgt = Critic(state_dim, train_cfg.critic_hidden).to(self.device)
        self.actor_tgt.load_state_dict(self.actor.state_dict())
        self.critic_tgt.load_state_dict(self.critic.state_dict())

        self.opt_actor = Adam(self.actor.parameters(), lr=train_cfg.lr_actor)
        self.opt_critic = Adam(self.critic.parameters(), lr=train_cfg.lr_critic)

    def pretrain_filter(self) -> None:
        trainer = ProbPretrainer(self.filter_model, self.simulator, self.tcfg)
        trainer.train()

    def build_state(self, batch: Dict[str, torch.Tensor], next_window: bool = False) -> torch.Tensor:
        x = batch["X_tp1"] if next_window else batch["X_t"]
        s = batch["S_tp1"] if next_window else batch["S_t"]
        i = batch["I_tp1"] if next_window else batch["I_t"]

        with torch.no_grad():
            probs = self.filter_model.probs(x)

        state = torch.cat([s, i, probs], dim=-1)
        self.normalizer.update(state.detach())
        return self.normalizer.normalize(state).detach()

    def train(self) -> None:
        self.pretrain_filter()
        self.filter_model.eval()

        for step in range(1, self.tcfg.train_steps_rl + 1):
            batch = self.simulator.sample_batch(self.tcfg.batch_size)

            state_t = self.build_state(batch, next_window=False)

            eps = exploration_noise(step, self.tcfg.exploration_a, self.tcfg.exploration_min, self.device)
            with torch.no_grad():
                a_noise = eps * torch.randn((self.tcfg.batch_size, 1), device=self.device)

                i_tp1 = self.actor(state_t) + a_noise
                i_tp1 = i_tp1.clamp(self.mcfg.i_min, self.mcfg.i_max)

            reward = trading_reward(
                i_tp1=i_tp1,
                i_t=batch["I_t"],
                s_t=batch["S_t"],
                s_tp1=batch["S_tp1"],
                lam=self.mcfg.lam,
            )

            batch["I_tp1"] = i_tp1.detach()

            with torch.no_grad():
                state_tp1 = self.build_state(batch, next_window=True)
                a_tp1 = self.actor_tgt(state_tp1).clamp(self.mcfg.i_min, self.mcfg.i_max)
                y = reward + self.mcfg.gamma * self.critic_tgt(state_tp1, a_tp1)

            for _ in range(self.tcfg.critic_updates_per_step):
                q = self.critic(state_t, i_tp1.detach())
                loss_critic = F.mse_loss(q, y)

                self.opt_critic.zero_grad()
                loss_critic.backward()
                self.opt_critic.step()

            for _ in range(self.tcfg.actor_updates_per_step):
                a = self.actor(state_t).clamp(self.mcfg.i_min, self.mcfg.i_max)
                loss_actor = -self.critic(state_t.detach(), a).mean()

                self.opt_actor.zero_grad()
                loss_actor.backward()
                self.opt_actor.step()

            soft_update(self.actor_tgt, self.actor, self.tcfg.tau_target)
            soft_update(self.critic_tgt, self.critic, self.tcfg.tau_target)

            if step % 200 == 0:
                print(
                    f"[prob-DDPG] step={step:4d} critic={loss_critic.item():.6f} "
                    f"actor={loss_actor.item():.6f} reward_mean={reward.mean().item():.4f}"
                )

        self.evaluate_policy(self.actor, self._eval_state_builder, "prob-DDPG")

    def _eval_state_builder(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        with torch.no_grad():
            probs = self.filter_model.probs(batch["X_t"])
        state = torch.cat([batch["S_t"], batch["I_t"], probs], dim=-1)
        return self.normalizer.normalize(state)

    def save(self, path: str) -> None:
        save_checkpoint(
            path,
            {
                "filter_model": self.filter_model.state_dict(),
                "actor": self.actor.state_dict(),
                "critic": self.critic.state_dict(),
                "actor_tgt": self.actor_tgt.state_dict(),
                "critic_tgt": self.critic_tgt.state_dict(),
                "opt_actor": self.opt_actor.state_dict(),
                "opt_critic": self.opt_critic.state_dict(),
                "market_cfg": self.mcfg.__dict__,
                "train_cfg": self.tcfg.__dict__,
            },
        )

    def load(self, path: str) -> None:
        ckpt = load_checkpoint(path, device=self.device)
        self.filter_model.load_state_dict(ckpt["filter_model"])
        self.actor.load_state_dict(ckpt["actor"])
        self.critic.load_state_dict(ckpt["critic"])
        self.actor_tgt.load_state_dict(ckpt["actor_tgt"])
        self.critic_tgt.load_state_dict(ckpt["critic_tgt"])
        self.opt_actor.load_state_dict(ckpt["opt_actor"])
        self.opt_critic.load_state_dict(ckpt["opt_critic"])

In [84]:
class RegDDPGTrainer(BaseDDPGTrainer):
    def __init__(self, market_cfg: MarketConfig, train_cfg: TrainingConfig):
        super().__init__(market_cfg, train_cfg)

        self.predictor = PriceRegressor(
            hidden_size=train_cfg.hidden_size_gru,
            num_layers=train_cfg.gru_layers,
        ).to(self.device)

        state_dim = 3  # S_t + I_t + S_hat_{t+1}
        self.normalizer = self._build_state_normalizer(state_dim)

        self.actor = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic = Critic(state_dim, train_cfg.critic_hidden).to(self.device)

        self.actor_tgt = Actor(state_dim, train_cfg.actor_hidden, market_cfg.i_max).to(self.device)
        self.critic_tgt = Critic(state_dim, train_cfg.critic_hidden).to(self.device)
        self.actor_tgt.load_state_dict(self.actor.state_dict())
        self.critic_tgt.load_state_dict(self.critic.state_dict())

        self.opt_actor = Adam(self.actor.parameters(), lr=train_cfg.lr_actor)
        self.opt_critic = Adam(self.critic.parameters(), lr=train_cfg.lr_critic)

    def pretrain_predictor(self) -> None:
        trainer = RegPretrainer(self.predictor, self.simulator, self.tcfg)
        trainer.train()

    def build_state(self, batch: Dict[str, torch.Tensor], next_window: bool = False) -> torch.Tensor:
        x = batch["X_tp1"] if next_window else batch["X_t"]
        s = batch["S_tp1"] if next_window else batch["S_t"]
        i = batch["I_tp1"] if next_window else batch["I_t"]

        with torch.no_grad():
            pred = self.predictor(x)

        state = torch.cat([s, i, pred], dim=-1)
        self.normalizer.update(state.detach())
        return self.normalizer.normalize(state).detach()

    def train(self) -> None:
        self.pretrain_predictor()
        self.predictor.eval()

        for step in range(1, self.tcfg.train_steps_rl + 1):
            batch = self.simulator.sample_batch(self.tcfg.batch_size)

            state_t = self.build_state(batch, next_window=False)

            eps = exploration_noise(step, self.tcfg.exploration_a, self.tcfg.exploration_min, self.device)
            with torch.no_grad():
                a_noise = eps * torch.randn((self.tcfg.batch_size, 1), device=self.device)

                i_tp1 = self.actor(state_t) + a_noise
                i_tp1 = i_tp1.clamp(self.mcfg.i_min, self.mcfg.i_max)

            reward = trading_reward(
                i_tp1=i_tp1,
                i_t=batch["I_t"],
                s_t=batch["S_t"],
                s_tp1=batch["S_tp1"],
                lam=self.mcfg.lam,
            )

            batch["I_tp1"] = i_tp1.detach()

            with torch.no_grad():
                state_tp1 = self.build_state(batch, next_window=True)
                a_tp1 = self.actor_tgt(state_tp1).clamp(self.mcfg.i_min, self.mcfg.i_max)
                y = reward + self.mcfg.gamma * self.critic_tgt(state_tp1, a_tp1)

            for _ in range(self.tcfg.critic_updates_per_step):
                q = self.critic(state_t, i_tp1.detach())
                loss_critic = F.mse_loss(q, y)

                self.opt_critic.zero_grad()
                loss_critic.backward()
                self.opt_critic.step()

            for _ in range(self.tcfg.actor_updates_per_step):
                a = self.actor(state_t).clamp(self.mcfg.i_min, self.mcfg.i_max)
                loss_actor = -self.critic(state_t.detach(), a).mean()

                self.opt_actor.zero_grad()
                loss_actor.backward()
                self.opt_actor.step()

            soft_update(self.actor_tgt, self.actor, self.tcfg.tau_target)
            soft_update(self.critic_tgt, self.critic, self.tcfg.tau_target)

            if step % 200 == 0:
                print(
                    f"[reg-DDPG]  step={step:4d} critic={loss_critic.item():.6f} "
                    f"actor={loss_actor.item():.6f} reward_mean={reward.mean().item():.4f}"
                )

        self.evaluate_policy(self.actor, self._eval_state_builder, "reg-DDPG")

    def _eval_state_builder(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        with torch.no_grad():
            pred = self.predictor(batch["X_t"])
        state = torch.cat([batch["S_t"], batch["I_t"], pred], dim=-1)
        return self.normalizer.normalize(state)

    def save(self, path: str) -> None:
        save_checkpoint(
            path,
            {
                "predictor": self.predictor.state_dict(),
                "actor": self.actor.state_dict(),
                "critic": self.critic.state_dict(),
                "actor_tgt": self.actor_tgt.state_dict(),
                "critic_tgt": self.critic_tgt.state_dict(),
                "opt_actor": self.opt_actor.state_dict(),
                "opt_critic": self.opt_critic.state_dict(),
                "market_cfg": self.mcfg.__dict__,
                "train_cfg": self.tcfg.__dict__,
            },
        )

    def load(self, path: str) -> None:
        ckpt = load_checkpoint(path, device=self.device)
        self.predictor.load_state_dict(ckpt["predictor"])
        self.actor.load_state_dict(ckpt["actor"])
        self.critic.load_state_dict(ckpt["critic"])
        self.actor_tgt.load_state_dict(ckpt["actor_tgt"])
        self.critic_tgt.load_state_dict(ckpt["critic_tgt"])
        self.opt_actor.load_state_dict(ckpt["opt_actor"])
        self.opt_critic.load_state_dict(ckpt["opt_critic"])

In [87]:
def main():
    set_seed(123)

    market_cfg = MarketConfig(
        dt=0.2,
        gamma=0.99,
        lam=0.05,
        i_min=-10.0,
        i_max=10.0,
        window=10,
        device="cuda",
        scenario="theta",
    )

    train_cfg = TrainingConfig(
        batch_size=256,
        train_steps_supervised=1500,
        train_steps_rl=2500,
        test_episodes=100,
        episode_length=300,
        lr_supervised=1e-3,
        lr_actor=1e-3,
        lr_critic=1e-3,
        lr_aux=1e-3,
        hidden_size_gru=20,
        gru_layers=1,
        actor_hidden=128,
        critic_hidden=128,
        tau_target=1e-3,
        exploration_a=100.0,
        exploration_min=0.05,
    )

    save_dir = "checkpoints"

    print("\n================ HID-DDPG ================\n")
    hid_trainer = HidDDPGTrainer(market_cfg, train_cfg)
    hid_trainer.train()
    hid_trainer.save(f"{save_dir}/hidg_final.pt")

    print("\n================ PROB-DDPG ================\n")
    prob_trainer = ProbDDPGTrainer(market_cfg, train_cfg)
    prob_trainer.train()

    prob_trainer.save(f"{save_dir}/prob_ddpg_final.pt")

    print("\n================ REG-DDPG ================\n")
    reg_trainer = RegDDPGTrainer(market_cfg, train_cfg)
    reg_trainer.train()

    reg_trainer.save(f"{save_dir}/reg_ddpg_final.pt")

In [88]:
main()


================ HID-DDPG ================

[hid-DDPG] step= 200 aux=0.014145 critic=2.039882 actor=0.717261 reward_mean=-0.5112
[hid-DDPG] step= 400 aux=0.014097 critic=0.531999 actor=-0.130547 reward_mean=0.1790
[hid-DDPG] step= 600 aux=0.013516 critic=0.945218 actor=-0.102379 reward_mean=0.2551
[hid-DDPG] step= 800 aux=0.012269 critic=1.034724 actor=-0.156084 reward_mean=0.3432
[hid-DDPG] step=1000 aux=0.012972 critic=0.791374 actor=-0.108493 reward_mean=0.2245
[hid-DDPG] step=1200 aux=0.014085 critic=0.842049 actor=-0.097749 reward_mean=0.2525
[hid-DDPG] step=1400 aux=0.013846 critic=0.801691 actor=-0.283633 reward_mean=0.3167
[hid-DDPG] step=1600 aux=0.010838 critic=0.613679 actor=-0.257226 reward_mean=0.3087
[hid-DDPG] step=1800 aux=0.012726 critic=0.576595 actor=-0.302541 reward_mean=0.2229
[hid-DDPG] step=2000 aux=0.013922 critic=0.756460 actor=-0.307796 reward_mean=0.2104
[hid-DDPG] step=2200 aux=0.009765 critic=0.403650 actor=-0.376554 reward_mean=0.2285
[hid-DDPG] step=2400